# Proyecto de Recuperación de Información


## Análisis y Procesamiento de Corpus de Documentos
- Carga de datos
- Limpieza y exploración
- Preprocesamiento (tokenización, eliminación de stopwords, normalización, stemming)


In [25]:
# Instalación de dependencias
#!pip install nltk scikit-learn numpy pandas kagglehub

### Sección 1: Configuración e Instalación


In [26]:
from pathlib import Path
import pandas as pd
import unicodedata
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [27]:
# Descargar recursos de NLTK
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

True

### Sección 2: Carga de Datos


In [28]:
# Descarga del dataset (comentado si ya existe)
import kagglehub

# Download latest version into ./data
data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)
path = kagglehub.dataset_download(
    "thedevastator/uncovering-financial-insights-with-the-reuters-2",
    output_dir=str(data_dir),
)

print("Path to dataset files:", path)

Path to dataset files: data


In [29]:
for file in data_dir.iterdir():
    print(f"  - {file.name}")


  - .complete
  - ModApte_test.csv
  - ModApte_train.csv
  - ModApte_unused.csv
  - ModHayes_test.csv
  - ModHayes_train.csv
  - ModLewis_test.csv
  - ModLewis_train.csv
  - ModLewis_unused.csv


In [30]:
import os
import csv  # <--- CORRECCIÓN: Aquí importamos la librería que faltaba

def load_all_corpus_csv(corpus_path):
    """
    Escanea la carpeta data y lee de forma masiva todos los archivos CSV fila por fila.
    """
    all_rows = []
    if not os.path.exists(corpus_path):
        print(f"Error: La ruta '{corpus_path}' no existe.")
        return pd.DataFrame(columns=["doc_id", "title", "text"])

    for file_name in os.listdir(corpus_path):
        # Buscamos todos los archivos .csv sueltos en la carpeta data
        if file_name.lower().endswith('.csv'):
            file_path = os.path.join(corpus_path, file_name)
            dataset_name = os.path.splitext(file_name)[0]

            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    reader = csv.reader(f)
                    for i, row in enumerate(reader):
                        if not row or len(row) == 0:
                            continue
                        # ID único para cada documento combinando el dataset y la fila
                        doc_id = f"{dataset_name}_doc_{i+1}"
                        text_content = " ".join(row)
                        # Estructuramos en columnas compatibles con tu limpieza de datos
                        all_rows.append({"doc_id": doc_id, "title": "", "text": text_content})
            except Exception as e:
                print(f"Error en archivo {file_name}: {e}")

    df = pd.DataFrame(all_rows)
    print("=" * 100)
    print(f"Dimensión total del dataset completo: {df.shape}")
    print("=" * 100)
    return df

# CORRECCIÓN DE RUTA: Como tus archivos están directo en 'data', apuntamos a esa carpeta
ruta_corpus = 'data'
corpus = load_all_corpus_csv(ruta_corpus)

Dimensión total del dataset completo: (55745, 3)


### Sección 3: Exploración de Datos

In [31]:
print("Vista previa de los datos:")
corpus.head(10)

Vista previa de los datos:


,doc_id,title,text
0,ModApte_test_doc_1,,text text_type topics lewis_split cgis_split o...
1,ModApte_test_doc_2,,Mounting trade friction between the\nU.S. And ...
2,ModApte_test_doc_3,,A survey of 19 provinces and seven cities\nsho...
3,ModApte_test_doc_4,,The Ministry of International Trade and\nIndus...
4,ModApte_test_doc_5,,Thailand's trade deficit widened to 4.5\nbilli...
5,ModApte_test_doc_6,,Indonesia expects crude palm oil (CPO)\nprices...
6,ModApte_test_doc_7,,"""BRIEF"" [] ""TEST"" ""TRAINING-SET"" ""3818"" ""1483..."
7,ModApte_test_doc_8,,"Tug crews in New South Wales (NSW),\nVictoria ..."
8,ModApte_test_doc_9,,The Indonesian Commodity Exchange is\nlikely t...
9,ModApte_test_doc_10,,Food Department officials said the U.S.\nDepar...


## Sección 4: Limpieza de Datos

In [32]:
def clean_dataset(df):
    """
    Limpia el dataset: rellena valores nulos, limpia columnas y crea campo de documento.
    """

    df = df.copy()

    # Completar valores nulos
    df["title"] = df["title"].fillna("")
    df["text"] = df["text"].fillna("")

    # Limpiar columnas innecesarias
    cols = ["text_type", "lewis_split", "cgis_split"]

    for col in cols:

        if col in df.columns:

            df[col] = (
                df[col]
                .astype(str)
                .str.replace('"', '', regex=False)
            )

    # Crear documento combinado
    df["document"] = (
        df["title"].astype(str)
        + " " +
        df["text"].astype(str)
    )

    # Limpiar espacios extras
    df["document"] = (
        df["document"]
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )

    # Eliminar documentos vacíos
    df = df[
        df["document"].str.strip() != ""
    ]

    print(f"Nuevo tamaño del dataset: {df.shape}")

    return df

In [33]:
corpus = clean_dataset(corpus)


Nuevo tamaño del dataset: (55745, 4)


In [34]:
print("Dataset limpio - Primeras filas:")
corpus.head(10)

Dataset limpio - Primeras filas:


,doc_id,title,text,document
0,ModApte_test_doc_1,,text text_type topics lewis_split cgis_split o...,text text_type topics lewis_split cgis_split o...
1,ModApte_test_doc_2,,Mounting trade friction between the\nU.S. And ...,Mounting trade friction between the U.S. And J...
2,ModApte_test_doc_3,,A survey of 19 provinces and seven cities\nsho...,A survey of 19 provinces and seven cities show...
3,ModApte_test_doc_4,,The Ministry of International Trade and\nIndus...,The Ministry of International Trade and Indust...
4,ModApte_test_doc_5,,Thailand's trade deficit widened to 4.5\nbilli...,Thailand's trade deficit widened to 4.5 billio...
5,ModApte_test_doc_6,,Indonesia expects crude palm oil (CPO)\nprices...,Indonesia expects crude palm oil (CPO) prices ...
6,ModApte_test_doc_7,,"""BRIEF"" [] ""TEST"" ""TRAINING-SET"" ""3818"" ""1483...","""BRIEF"" [] ""TEST"" ""TRAINING-SET"" ""3818"" ""14835..."
7,ModApte_test_doc_8,,"Tug crews in New South Wales (NSW),\nVictoria ...","Tug crews in New South Wales (NSW), Victoria a..."
8,ModApte_test_doc_9,,The Indonesian Commodity Exchange is\nlikely t...,The Indonesian Commodity Exchange is likely to...
9,ModApte_test_doc_10,,Food Department officials said the U.S.\nDepar...,Food Department officials said the U.S. Depart...


### Sección 5: Pipeline de Preprocesamiento

El preprocesamiento incluye:
1. **Tokenización**: Dividir documentos en palabras
2. **Eliminación de stopwords**: Remover palabras comunes sin valor semántico
3. **Normalización**: Convertir a minúsculas, quitar acentos, eliminar caracteres especiales
4. **Stemming**: Reducir palabras a su raíz

#### 5.1 Tokenización

In [35]:
def tokenize(corpus):
    """
    Tokeniza cada documento en el corpus en palabras individuales.

    Args:
        corpus (pd.DataFrame): DataFrame con documentos

    Returns:
        pd.DataFrame: DataFrame con columna 'tokens'
    """
    # Tokenizar cada documento
    corpus["tokens"] = (corpus["document"].apply(word_tokenize))

    print(f"Tokens creados. Ejemplo (primeros 10): {corpus['tokens'].iloc[0][:10]}")
    return corpus


#### 5.2 Eliminación de Stopwords

In [36]:
def quit_stopwords(corpus):
    """
    Elimina stopwords (palabras comunes) del corpus y convierte a minúsculas.

    Args:
        corpus (pd.DataFrame): DataFrame con tokens

    Returns:
        pd.DataFrame: DataFrame con columna 'no_stopwords'
    """
    stop_words = set(stopwords.words("english"))

    # Poner en minúsculas
    corpus["no_stopwords"] = (
        corpus["tokens"]
        .apply(
            lambda words: [
                word.lower()
                for word in words
            ]
        )
    )

    # Quitar las stopwords
    corpus["no_stopwords"] = (
        corpus["no_stopwords"]
        .apply(
            lambda words: [
                word
                for word in words
                if word not in stop_words
            ]
        )
    )

    print(f"Stopwords removidos. Ejemplo: {corpus['no_stopwords'].iloc[0][:10]}")
    return corpus


#### 5.3 Normalización

In [37]:
def normalize(tokens):
    """
    Normaliza tokens: convierte a minúsculas, elimina acentos,
    y mantiene solo caracteres alfabéticos.

    Args:
        tokens (list): Lista de tokens

    Returns:
        list: Lista de tokens normalizados
    """
    normalized = []

    for token in tokens:
        # lowercase (case folding)
        token = token.lower()

        # quitar acentos
        token = unicodedata.normalize(
            "NFKD",
            token
        ).encode(
            "ascii",
            "ignore"
        ).decode("utf-8")

        # dejar solo letras
        token = re.sub(
            r"[^a-z]",
            "",
            token
        )

        # evitar vacíos
        if token:
            normalized.append(token)

    return normalized


#### 5.4 Stemming

In [38]:
def stemming(corpus):
    """
    Aplica stemming (reducción a raíz) a los tokens normalizados.

    Args:
        corpus (pd.DataFrame): DataFrame con columna 'normalized'

    Returns:
        pd.DataFrame: DataFrame con columna 'stemmed'
    """
    stemmer = PorterStemmer()

    corpus["stemmed"] = (
        corpus["normalized"]
        .apply(
            lambda words: [
                stemmer.stem(word)
                for word in words
            ]
        )
    )

    print(f"Stemming aplicado. Ejemplo: {corpus['stemmed'].iloc[0][:10]}")
    return corpus

#### 5.5 Pipeline Completo de Preprocesamiento

In [39]:
def preprocessing(corpus):
    """
    Ejecuta el pipeline completo de preprocesamiento:
    tokenización → eliminación de stopwords → normalización → stemming

    Args:
        corpus (pd.DataFrame): DataFrame con documentos originales

    Returns:
        pd.DataFrame: DataFrame preprocesado con columnas de tokens, stopwords, normalized, y stemmed
    """
    print("Iniciando pipeline de preprocesamiento...\n")

    # Tokenizar
    print("1. Tokenización...")
    corpus = tokenize(corpus)

    # Quitar stopwords
    print("\n2. Eliminación de stopwords...")
    corpus = quit_stopwords(corpus)

    # Normalización (case folding, acentos, etc.)
    print("\n3. Normalización...")
    corpus["normalized"] = (corpus["no_stopwords"].apply(normalize))

    # Stemming
    print("\n4. Stemming...")
    corpus = stemming(corpus)

    print("\nPipeline completado exitosamente")
    return corpus


### Sección 6: Ejecución del Pipeline

In [40]:
print("Procesando corpus...")
corpus = preprocessing(corpus)

Procesando corpus...
Iniciando pipeline de preprocesamiento...

1. Tokenización...
Tokens creados. Ejemplo (primeros 10): ['text', 'text_type', 'topics', 'lewis_split', 'cgis_split', 'old_id', 'new_id', 'places', 'people', 'orgs']

2. Eliminación de stopwords...
Stopwords removidos. Ejemplo: ['text', 'text_type', 'topics', 'lewis_split', 'cgis_split', 'old_id', 'new_id', 'places', 'people', 'orgs']

3. Normalización...

4. Stemming...
Stemming aplicado. Ejemplo: ['text', 'texttyp', 'topic', 'lewissplit', 'cgissplit', 'oldid', 'newid', 'place', 'peopl', 'org']

Pipeline completado exitosamente


### Sección 7: Resultados

In [41]:
print("Corpus preprocesado - Información final:")
print(f"Tamaño del corpus: {corpus.shape}")
print(f"\nColumnas disponibles: {list(corpus.columns)}")

Corpus preprocesado - Información final:
Tamaño del corpus: (55745, 8)

Columnas disponibles: ['doc_id', 'title', 'text', 'document', 'tokens', 'no_stopwords', 'normalized', 'stemmed']


In [42]:
print("Ejemplo de tokens procesados (primeros 100 elementos del primer documento):")
print(corpus["stemmed"].iloc[0][:100])

Ejemplo de tokens procesados (primeros 100 elementos del primer documento):
['text', 'texttyp', 'topic', 'lewissplit', 'cgissplit', 'oldid', 'newid', 'place', 'peopl', 'org', 'exchang', 'date', 'titl']


In [43]:
print("\nEstadísticas de tokenización y preprocesamiento:")
print(f"Promedio de tokens por documento: {corpus['tokens'].apply(len).mean():.2f}")
print(f"Promedio de tokens sin stopwords: {corpus['no_stopwords'].apply(len).mean():.2f}")
print(f"Promedio de tokens finales (después de stemming): {corpus['stemmed'].apply(len).mean():.2f}")



Estadísticas de tokenización y preprocesamiento:
Promedio de tokens por documento: 177.84
Promedio de tokens sin stopwords: 135.34
Promedio de tokens finales (después de stemming): 88.58


In [44]:
corpus[["document", "stemmed"]].head(5)

,document,stemmed
0,text text_type topics lewis_split cgis_split o...,"[text, texttyp, topic, lewissplit, cgissplit, ..."
1,Mounting trade friction between the U.S. And J...,"[mount, trade, friction, us, japan, rais, fear..."
2,A survey of 19 provinces and seven cities show...,"[survey, provinc, seven, citi, show, vermin, c..."
3,The Ministry of International Trade and Indust...,"[ministri, intern, trade, industri, miti, revi..."
4,Thailand's trade deficit widened to 4.5 billio...,"[thailand, s, trade, deficit, widen, billion, ..."


### Sección 8: Vocabulario e Índice Invertido

In [45]:
from collections import defaultdict

# Crear vocabulario
vocabulario = set()

for tokens in corpus["stemmed"]:
    vocabulario.update(tokens)

print("Cantidad de palabras únicas:")
print(len(vocabulario))

Cantidad de palabras únicas:
37982


## Construcción del índice

• Construcción de un índice invertido que almacene, para cada término, los documentos en los que
aparece y su frecuencia

In [46]:
import math
def build_inverted_index(df):
    """
    Construye el Índice Invertido: { término: { doc_id: frecuencia } }
    """
    inverted_index = {}
    for _, row in df.iterrows():
        doc_id = row["doc_id"]
        words = row["stemmed"]  # Usamos las palabras limpias finales con stemming

        for word in words:
            if word not in inverted_index:
                inverted_index[word] = {}
            # Guardamos el ID del documento y sumamos 1 a su frecuencia local
            inverted_index[word][doc_id] = inverted_index[word].get(doc_id, 0) + 1
    return inverted_index

In [47]:
inverted_index = build_inverted_index(corpus)

In [48]:
print(f"Indice invertido de market:{inverted_index['switzerland']}")
# print(inverted_index["stocks"])
# print(inverted_index["oil"])

Indice invertido de market:{'ModApte_test_doc_28': 1, 'ModApte_test_doc_48': 1, 'ModApte_test_doc_50': 1, 'ModApte_test_doc_108': 1, 'ModApte_test_doc_348': 1, 'ModApte_test_doc_365': 1, 'ModApte_test_doc_376': 1, 'ModApte_test_doc_377': 1, 'ModApte_test_doc_379': 1, 'ModApte_test_doc_402': 1, 'ModApte_test_doc_407': 1, 'ModApte_test_doc_408': 1, 'ModApte_test_doc_418': 1, 'ModApte_test_doc_425': 1, 'ModApte_test_doc_431': 1, 'ModApte_test_doc_450': 1, 'ModApte_test_doc_467': 2, 'ModApte_test_doc_543': 1, 'ModApte_test_doc_632': 1, 'ModApte_test_doc_838': 1, 'ModApte_test_doc_878': 4, 'ModApte_test_doc_917': 1, 'ModApte_test_doc_954': 2, 'ModApte_test_doc_1011': 1, 'ModApte_test_doc_1016': 1, 'ModApte_test_doc_1091': 1, 'ModApte_test_doc_1136': 1, 'ModApte_test_doc_1141': 2, 'ModApte_test_doc_1626': 2, 'ModApte_test_doc_1748': 1, 'ModApte_test_doc_2007': 1, 'ModApte_test_doc_2048': 1, 'ModApte_test_doc_2123': 1, 'ModApte_test_doc_3185': 1, 'ModApte_train_doc_7': 1, 'ModApte_train_doc_1

## Recuperación

### Recuperación basada en similitud Jaccard utilizando vectores binarios

In [49]:
def preprocess(text):
    import unicodedata
    import re
    from nltk.tokenize import word_tokenize
    from nltk.corpus import stopwords
    from nltk.stem import PorterStemmer
    
    tokens = word_tokenize(text.lower())
    stop_words = set(stopwords.words("english"))
    tokens = [t for t in tokens if t not in stop_words]
    
    normalized = []
    for token in tokens:
        token = unicodedata.normalize("NFKD", token).encode("ascii", "ignore").decode("utf-8")
        token = re.sub(r"[^a-z]", "", token)
        if token:
            normalized.append(token)
    
    stemmer = PorterStemmer()
    stemmed = [stemmer.stem(token) for token in normalized]
    return stemmed

In [50]:
def jaccard_similarity(set1, set2):
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union if union != 0 else 0

In [51]:
def jaccard_search(query, top_k=10):
    """
    Búsqueda usando similitud Jaccard con vectores binarios
    
    Args:
        query (str): Consulta de texto libre
        top_k (int): Número de resultados a retornar
    
    Returns:
        list: Lista de tuplas (doc_id, score) ordenadas por relevancia
    """
    # Preprocesar la consulta
    query_tokens = preprocess(query)
    query_terms = set(query_tokens)
    
    if not query_terms:
        return []
    
    results = []
    for idx, row in corpus.iterrows():
        doc_terms = set(row["stemmed"])
        score = jaccard_similarity(query_terms, doc_terms)
        results.append((row["doc_id"], score))
    
    # Ordenar por score descendente
    results.sort(key=lambda x: x[1], reverse=True)
    
    return results[:top_k]

In [52]:
# Probar Jaccard
print("Resultados Jaccard para 'oil price crude':")
results_jaccard = jaccard_search("oil price crude")
for doc_id, score in results_jaccard[:5]:
    print(f"  {doc_id}: {score:.4f}")

Resultados Jaccard para 'oil price crude':
  ModApte_test_doc_2945: 0.2308
  ModApte_train_doc_2211: 0.2308
  ModHayes_train_doc_3355: 0.2308
  ModHayes_train_doc_20223: 0.2308
  ModLewis_test_doc_5555: 0.2308


## Implementacion TF-IDF 

In [53]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Preparar los documentos (usar los tokens stemmed unidos en strings)
corpus["text_stemmed"] = corpus["stemmed"].apply(lambda tokens: " ".join(tokens))

# Crear el vectorizador TF-IDF
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus["text_stemmed"])

print(f"Matriz TF-IDF creada: {tfidf_matrix.shape}")
print(f"Tamaño del vocabulario: {len(tfidf_vectorizer.get_feature_names_out())}")

Matriz TF-IDF creada: (55745, 37956)
Tamaño del vocabulario: 37956


## Funcion Busqueda TF-IDF

In [54]:
def search_tfidf(query, top_k=10):
    """
    Búsqueda usando TF-IDF y similitud de coseno
    """
    # Preprocesar la consulta
    query_tokens = preprocess(query)
    query_text = " ".join(query_tokens)
    
    # Transformar consulta a vector TF-IDF
    query_vector = tfidf_vectorizer.transform([query_text])
    
    # Calcular similitud coseno
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()
    
    # Obtener top_k resultados
    results = []
    for idx in similarities.argsort()[-top_k:][::-1]:
        results.append((corpus.iloc[idx]["doc_id"], similarities[idx]))
    
    return results

## Probar TF-IDF

In [55]:
# Probar TF-IDF
print("Resultados TF-IDF para 'oil price crude':")
results_tfidf = search_tfidf("oil price crude")
for doc_id, score in results_tfidf[:5]:
    print(f"  {doc_id}: {score:.4f}")

Resultados TF-IDF para 'oil price crude':
  ModHayes_train_doc_18025: 0.5780
  ModLewis_test_doc_3357: 0.5780
  ModApte_test_doc_1974: 0.5780
  ModApte_train_doc_91: 0.5543
  ModLewis_train_doc_128: 0.5543


## Instalar e importar Librerias de BM25

In [56]:
# !pip install rank-bm25 -q

import numpy as np
import math
from sklearn.feature_extraction.text import CountVectorizer

## Preparación del coorpus, matriz, longitud, y parametros

In [57]:
# 1. Preparar el corpus con stemming
documentos = corpus["text_stemmed"].tolist()

# 2. Crear matriz de frecuencias
vectorizer = CountVectorizer()
tf_matrix = vectorizer.fit_transform(documentos)
terminos = vectorizer.get_feature_names_out()

# 3. Longitudes de documentos
longitudes = np.array(tf_matrix.sum(axis=1)).flatten()
avg_longitud = np.mean(longitudes)

# 4. Calcular IDF (versión BM25)
N = len(documentos)
df = np.array((tf_matrix > 0).sum(axis=0)).flatten()
idf = np.log((N - df + 0.5) / (df + 0.5) + 1)

# 5. Parámetros BM25 (valores estándar)
k1 = 1.5
b = 0.75

## Función de búsqueda BM25

In [58]:
def search_bm25(query, top_k=10):
    """
    Implementación propia de BM25
    """
    # Preprocesar consulta
    query_tokens = preprocess(query)
    if not query_tokens:
        return []
    
    scores = np.zeros(len(documentos))
    
    # Calcular score para cada documento
    for i in range(len(documentos)):
        doc_len = longitudes[i]
        score = 0
        
        for token in set(query_tokens):
            if token in terminos:
                idx = np.where(terminos == token)[0][0]
                tf = tf_matrix[i, idx]
                
                if tf > 0:
                    # Fórmula BM25
                    numerador = tf * (k1 + 1)
                    denominador = tf + k1 * (1 - b + b * (doc_len / avg_longitud))
                    score += idf[idx] * (numerador / denominador)
        
        scores[i] = score
    
    # Obtener top_k resultados
    results = []
    for idx in scores.argsort()[-top_k:][::-1]:
        if scores[idx] > 0:
            results.append((corpus.iloc[idx]["doc_id"], float(scores[idx])))
    
    return results


## Probar BM25

In [59]:
print("Resultados BM25 para 'oil price crude':")
results_bm25 = search_bm25("oil price crude")
for doc_id, score in results_bm25[:5]:
    print(f"  {doc_id}: {score:.4f}")

Resultados BM25 para 'oil price crude':
  ModApte_test_doc_1987: 15.0152
  ModHayes_train_doc_18052: 15.0152
  ModLewis_test_doc_3384: 15.0152
  ModLewis_train_doc_128: 14.9509
  ModHayes_train_doc_128: 14.9509


## Comparar los 3 modelos

In [60]:
query_test = "oil price crude"

print("=" * 70)
print(f"COMPARACIÓN DE MODELOS (todos implementación propia)")
print("=" * 70)

print("\n🔹 JACCARD (binario):")
for doc_id, score in jaccard_search(query_test)[:5]:
    print(f"   {doc_id}: {score:.4f}")

print("\n🔹 TF-IDF + COSENO:")
for doc_id, score in search_tfidf(query_test)[:5]:
    print(f"   {doc_id}: {score:.4f}")

print("\n🔹 BM25:")
for doc_id, score in search_bm25(query_test)[:5]:
    print(f"   {doc_id}: {score:.4f}")

COMPARACIÓN DE MODELOS (todos implementación propia)

🔹 JACCARD (binario):
   ModApte_test_doc_2945: 0.2308
   ModApte_train_doc_2211: 0.2308
   ModHayes_train_doc_3355: 0.2308
   ModHayes_train_doc_20223: 0.2308
   ModLewis_test_doc_5555: 0.2308

🔹 TF-IDF + COSENO:
   ModHayes_train_doc_18025: 0.5780
   ModLewis_test_doc_3357: 0.5780
   ModApte_test_doc_1974: 0.5780
   ModApte_train_doc_91: 0.5543
   ModLewis_train_doc_128: 0.5543

🔹 BM25:
   ModApte_test_doc_1987: 15.0152
   ModHayes_train_doc_18052: 15.0152
   ModLewis_test_doc_3384: 15.0152
   ModLewis_train_doc_128: 14.9509
   ModHayes_train_doc_128: 14.9509


## EMBEDDINGS

## Instalar librerías necesarias En este caso el Chromadb

In [61]:
# !pip install sentence-transformers chromadb faiss-cpu -q

In [62]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils import embedding_functions
import faiss

## Cargar modelo de embeddings 

In [63]:
# El profe usaba: model = api.load("word2vec-google-news-300")
# Nosotros usamos un modelo específico para frases/documentos

print("Cargando modelo SentenceTransformer...")
modelo_embeddings = SentenceTransformer('all-MiniLM-L6-v2')
print("¡Modelo cargado!")
print(f"Dimensiones del embedding: {modelo_embeddings.get_sentence_embedding_dimension()}")

Cargando modelo SentenceTransformer...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\fredd\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\fredd\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

¡Modelo cargado!
Dimensiones del embedding: 384


C:\Users\fredd\AppData\Local\Temp\ipykernel_23060\854689163.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Dimensiones del embedding: {modelo_embeddings.get_sentence_embedding_dimension()}")


## Probar el modelo con una frase

In [64]:
# El profe hacía: model["dog"]
# Nosotros hacemos:

frase_ejemplo = "The price of crude oil increased today"
embedding = modelo_embeddings.encode(frase_ejemplo)

print(f"Frase: '{frase_ejemplo}'")
print(f"Embedding generado (primeros 10 valores): {embedding[:10]}")
print(f"Forma del embedding: {embedding.shape}")

Frase: 'The price of crude oil increased today'
Embedding generado (primeros 10 valores): [-0.05531029 -0.01444545  0.04314071  0.03075164  0.02999191 -0.01799647
 -0.06222944  0.06893652  0.01572975 -0.02931233]
Forma del embedding: (384,)


## Calcular similitud entre frases

In [65]:
from sklearn.metrics.pairwise import cosine_similarity

# El profe hacía: cosine_similarity([vec("teacher")], [vec("student")])
# Nosotros hacemos:

frase1 = "Oil prices are rising"
frase2 = "Crude petroleum costs increasing"
frase3 = "The weather is nice today"

emb1 = modelo_embeddings.encode([frase1])
emb2 = modelo_embeddings.encode([frase2])
emb3 = modelo_embeddings.encode([frase3])

sim_12 = cosine_similarity(emb1, emb2)[0][0]
sim_13 = cosine_similarity(emb1, emb3)[0][0]

print(f"Similitud entre '{frase1}' y '{frase2}': {sim_12:.4f}")
print(f"Similitud entre '{frase1}' y '{frase3}': {sim_13:.4f}")


Similitud entre 'Oil prices are rising' y 'Crude petroleum costs increasing': 0.6360
Similitud entre 'Oil prices are rising' y 'The weather is nice today': 0.1366


## Generar embeddings para TODO el corpus

In [66]:
# Esto es equivalente a lo que el profe hacía con palabras,
# pero nosotros lo hacemos con documentos completos

print("Generando embeddings para todos los documentos del corpus...")
print(f"Total documentos: {len(corpus)}")

# Usar los textos preprocesados (con stemming)
textos = corpus["text_stemmed"].tolist()

# Generar embeddings (puede tomar varios minutos)
embeddings_docs = modelo_embeddings.encode(textos, show_progress_bar=True)

print(f"Embeddings generados: {embeddings_docs.shape}")

Generando embeddings para todos los documentos del corpus...
Total documentos: 55745


Batches:   0%|          | 0/1743 [00:00<?, ?it/s]

Embeddings generados: (55745, 384)


## Crear base vectorial con FAISS

In [67]:
# FAISS es una base vectorial eficiente
print("Creando índice FAISS...")

# Normalizar embeddings para similitud coseno
faiss.normalize_L2(embeddings_docs)

# Crear índice
dimension = embeddings_docs.shape[1]
indice_faiss = faiss.IndexFlatIP(dimension)  # Inner Product = coseno
indice_faiss.add(embeddings_docs)

print(f"Índice FAISS creado con {indice_faiss.ntotal} vectores")

Creando índice FAISS...
Índice FAISS creado con 55745 vectores


## Función de búsqueda con embeddings

In [68]:
def search_embedding(query, top_k=10):
    """
    Búsqueda semántica usando embeddings (similar a model.most_similar())
    """
    # Generar embedding de la consulta
    query_emb = modelo_embeddings.encode([query])
    
    # Normalizar
    faiss.normalize_L2(query_emb)
    
    # Buscar los más similares en FAISS
    scores, indices = indice_faiss.search(query_emb, top_k)
    
    # Formatear resultados
    results = []
    for i, idx in enumerate(indices[0]):
        doc_id = corpus.iloc[idx]["doc_id"]
        score = float(scores[0][i])
        results.append((doc_id, score))
    
    return results

# Probar
print("Resultados con EMBEDDINGS para 'oil price crude':")
results_emb = search_embedding("oil price crude")
for doc_id, score in results_emb[:5]:
    print(f"  {doc_id}: {score:.4f}")

Resultados con EMBEDDINGS para 'oil price crude':
  ModLewis_test_doc_4609: 0.6372
  ModHayes_train_doc_19277: 0.6372
  ModApte_test_doc_2499: 0.6372
  ModLewis_train_doc_11143: 0.6214
  ModHayes_train_doc_11150: 0.6214


## Comparar los 4 modelos

In [69]:
query_test = "oil price crude"

print("=" * 70)
print(f"COMPARACIÓN DE LOS 4 MODELOS para: '{query_test}'")
print("=" * 70)

print("\n🔹 JACCARD (vectores binarios):")
for doc_id, score in jaccard_search(query_test)[:5]:
    print(f"   {doc_id}: {score:.4f}")

print("\n🔹 TF-IDF + COSENO:")
for doc_id, score in search_tfidf(query_test)[:5]:
    print(f"   {doc_id}: {score:.4f}")

print("\n🔹 BM25:")
for doc_id, score in search_bm25(query_test)[:5]:
    print(f"   {doc_id}: {score:.4f}")

print("\n🔹 EMBEDDINGS (semántico - similar a Word2Vec del profe):")
for doc_id, score in search_embedding(query_test)[:5]:
    print(f"   {doc_id}: {score:.4f}")

COMPARACIÓN DE LOS 4 MODELOS para: 'oil price crude'

🔹 JACCARD (vectores binarios):
   ModApte_test_doc_2945: 0.2308
   ModApte_train_doc_2211: 0.2308
   ModHayes_train_doc_3355: 0.2308
   ModHayes_train_doc_20223: 0.2308
   ModLewis_test_doc_5555: 0.2308

🔹 TF-IDF + COSENO:
   ModHayes_train_doc_18025: 0.5780
   ModLewis_test_doc_3357: 0.5780
   ModApte_test_doc_1974: 0.5780
   ModApte_train_doc_91: 0.5543
   ModLewis_train_doc_128: 0.5543

🔹 BM25:
   ModApte_test_doc_1987: 15.0152
   ModHayes_train_doc_18052: 15.0152
   ModLewis_test_doc_3384: 15.0152
   ModLewis_train_doc_128: 14.9509
   ModHayes_train_doc_128: 14.9509

🔹 EMBEDDINGS (semántico - similar a Word2Vec del profe):
   ModLewis_test_doc_4609: 0.6372
   ModHayes_train_doc_19277: 0.6372
   ModApte_test_doc_2499: 0.6372
   ModLewis_train_doc_11143: 0.6214
   ModHayes_train_doc_11150: 0.6214
